## 0) Install & Import

In [ ]:
# Dependency management helper function
def ensure_installed(package, import_name=None):
    import importlib, sys
    name = import_name or package
    try:
        return importlib.import_module(name)
    except ImportError:
        !{sys.executable} -m pip install {package}
        return importlib.import_module(name)

# Install OR-Tools and data libraries
cp_model = ensure_installed("ortools", "ortools.sat.python.cp_model")
pd = ensure_installed("pandas")
np = ensure_installed("numpy")
calendar = ensure_installed("calendar")

# Import necessary libraries
import pandas as pd
import numpy as np
import calendar
import math
import os
from pathlib import Path
from datetime import datetime, timedelta, date
from collections import defaultdict
from ortools.sat.python import cp_model

# Set plot imports
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [12, 8]

## 1) Schedule Parameters -- Adjust This

In [ ]:
# ------------------------------------------------------------
# RESIDENTS
# ------------------------------------------------------------
# Format: List of resident initials (edit as needed)
# ------------------------------------------------------------

# Junior residents
junior_residents = ["ZZZ", "XXX", "WWW", "VVV"]

# All residents
residents = junior_residents

# Used as a final tie-breaker between otherwise-equivalent solutions.
# Leave as [""] if not using manual tie-breaking.
tiebreaker_residents = [""]

# Set name of attempted solution
solution_name = "junior_V1"

output_dir = Path(solution_name)
output_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# DATE RANGE
# ------------------------------------------------------------
start_date       = datetime(2026, 7, 5)
end_date         = datetime(2027, 7, 4)

buddy_call_start = datetime(2026, 7, 5)
buddy_call_end   = datetime(2026, 9, 7)

# ------------------------------------------------------------
# HOLIDAYS WITH SHIFT OVERRIDES
# ------------------------------------------------------------
# Format: 
#   date : shift_length
# ------------------------------------------------------------
holidays = {
    datetime(2026, 7, 3)    :   24, # Independence Day Observed
    datetime(2026, 7, 4)    :   24, # Independence Day
    datetime(2026, 7, 5)    :   24, # Day after Independence Day
    datetime(2026, 9, 4)    :   12, # Labor Day Friday
    datetime(2026, 9, 5)    :   24, # Labor Day Weekend
    datetime(2026, 9, 6)    :   24, # Labor Day Weekend
    datetime(2026, 9, 7)    :   24, # Labor Day
    datetime(2026, 11, 25)  :   12, # Day before Thanksgiving
    datetime(2026, 11, 26)  :   24, # Thanksgiving
    datetime(2026, 11, 27)  :   24, # Day after Thanksgiving
    datetime(2026, 11, 28)  :   24, # Thanksgiving Weekend
    datetime(2026, 11, 29)  :   24, # Thanksgiving Weekend
    datetime(2026, 12, 24)  :   12, # Christmas Eve
    datetime(2026, 12, 25)  :   24, # Christmas
    datetime(2026, 12, 26)  :   24, # Christmas Weekend
    datetime(2026, 12, 27)  :   24, # Christmas Weekend
    datetime(2026, 12, 31)  :   12, # New Year's Eve
    datetime(2027, 1, 1)    :   24, # New Year's Day
    datetime(2027, 1, 2)    :   24, # New Year's Weekend
    datetime(2027, 1, 3)    :   24, # New Year's Weekend
    datetime(2027, 1, 15)   :   12, # Martin Luther King Jr. Day Preceeding Friday
    datetime(2027, 1, 16)   :   24, # Martin Luther King Jr. Day Weekend
    datetime(2027, 1, 17)   :   24, # Martin Luther King Jr. Day Weekend
    datetime(2027, 1, 18)   :   24, # Martin Luther King Jr. Day
    datetime(2027, 2, 12)   :   12, # Presidents' Day
    datetime(2027, 2, 13)   :   24, # Presidents' Day
    datetime(2027, 2, 14)   :   24, # Presidents' Day
    datetime(2027, 2, 15)   :   24, # Presidents' Day
    datetime(2027, 5, 28)   :   12, # Memorial Day
    datetime(2027, 5, 29)   :   24, # Memorial Day
    datetime(2027, 5, 30)   :   24, # Memorial Day
    datetime(2027, 5, 31)   :   24, # Memorial Day
    datetime(2027, 6, 17)   :   12, # Juneteenth Eve
    datetime(2027, 6, 18)   :   24, # Juneteenth Observed
    datetime(2027, 6, 19)   :   24, # Juneteenth Weekend
    datetime(2027, 6, 20)   :   24, # Juneteenth Weekend
    datetime(2027, 7, 3)    :   24, # Independence Day Observed
    datetime(2027, 7, 4)    :   24, # Independence Day
}

# ------------------------------------------------------------
# ROTATION BLOCK DEFINITIONS
# ------------------------------------------------------------
# Format:
#   {"block": block_number, "start": start_date, "end": end_date}
# ------------------------------------------------------------
rotation_blocks = [
    {"block": 1, "start": datetime(2026, 6, 11).date(),     "end": datetime(2026, 9, 10).date()},
    {"block": 2, "start": datetime(2026, 9, 11).date(),     "end": datetime(2026, 11, 10).date()},
    {"block": 3, "start": datetime(2026, 11, 11).date(),    "end": datetime(2027, 2, 10).date()},
    {"block": 4, "start": datetime(2027, 2, 11).date(),      "end": datetime(2027, 4, 10).date()},
    {"block": 5, "start": datetime(2027, 4, 11).date(),     "end": datetime(2027, 6, 10).date()},
]

# ------------------------------------------------------------
# ROTATION ASSIGNMENTS PER RESIDENT
# ------------------------------------------------------------
# Format:
#   "ResidentCode": {block_number: "RotationName", ...}
# ------------------------------------------------------------
# Block numbers here match the block-number keys in resident_rotations below.
resident_rotations = { 
    "ZZZ": {1: "Rotation 1", 2: "Rotation 2", 3: "Rotation 3", 4: "Rotation 4", 5: "Rotation 5"}
}

# ------------------------------------------------------------
# HALF-DAY DEFINITIONS PER ROTATION
# ------------------------------------------------------------
# Format: 0 = Monday, 1 = Tuesday, 2 = Wednesday, 3 = Thursday, 4 = Friday, 5 = Saturday, 6 = Sunday
#   "RotationName": {
#       "AM": [
#           {"weekday": 2, "nth": 2},   # 2nd Wednesday
#           {"weekday": 2, "nth": 4},   # 4th Wednesday
#       ],
#       "PM": [
#           {"weekday": 1, "nth": 3},   # 3rd Tuesday
#       ]
#   },
# ------------------------------------------------------------
rotation_half_days = {
    "Rotation 1":           {"AM": [],"PM": []},
    "Rotation 2":           {"AM": [],"PM": []},
    "Rotation 3":           {"AM": [],"PM": []},
    "Rotation 4":           {"AM": [],"PM": []},
    "Rotation 5":           {"AM": [],"PM": []}
}

# ------------------------------------------------------------
# OR-DAY DEFINITIONS PER ROTATION
# ------------------------------------------------------------
# Format: 0 = Monday, 1 = Tuesday, 2 = Wednesday, 3 = Thursday, 4 = Friday, 5 = Saturday, 6 = Sunday
#   "RotationName": {
#       "AM": [
#           {"weekday": 2, "nth": 2},   # 2nd Wednesday
#           {"weekday": 2, "nth": 4},   # 4th Wednesday
#       ],
#       "PM": [
#           {"weekday": 1, "nth": 3},   # 3rd Tuesday
#       ]
#   },
# ------------------------------------------------------------
rotation_or_days = {
    "Rotation 1":           [],
    "Rotation 2":           [],
    "Rotation 3":           [],
    "Rotation 4":           [],
    "Rotation 5":           [] 
}

# ------------------------------------------------------------
# VACATIONS (per employee)
# ------------------------------------------------------------
# Format:
#   "Initials": [(start_date, end_date), (start_date, end_date), ...]
# NOTE: Flanking weekends and holidays are added later
# ------------------------------------------------------------
vacation_ranges = [
    {"res": "ZZZ", "start": "2026-12-29", "end": "2027-01-02"},
]

# ------------------------------------------------------------
# RESIDENT PREFERENCES (for soft constraints)
# ------------------------------------------------------------
# Format:
#   "ResidentCode": {"avoid_ranges": [(start_date, end_date), ...], ...}
# -------------------------------------------------------------
resident_preferences = {
}

# ------------------------------------------------------------
# SHIFT TRANSFER DEFINITIONS
# ------------------------------------------------------------
# Format:
#   (giver, receiver, amount, hours_filter, buddy_filter, holiday_filter)
#
# hours_filter: 12, 24, or None
# buddy_filter: True (only buddy), False (non-buddy), None (any)
# holiday_filter: True, False, or None
shift_transfers = [
]

# ------------------------------------------------------------
# MANUAL ASSIGNMENTS (for specific dates, e.g. conferences, etc.)
# ------------------------------------------------------------
# Format:
#  date : "ResidentName" or [list of resident names]
# ------------------------------------------------------------
manual_assignments = {
}

### Optional: Import prior schedule as manual assignments

In [ ]:
def load_manual_assignments_from_csv(csv_path, existing_manual=None, start_date=None, end_date=None):
    """
    Load manual assignments from a 'long list' CSV and merge with existing manual assignments.

    Expected CSV columns:
        Date | Resident | Holiday | VacationingResidents | ORDayResidents

    Returns:
        dict[date -> list of residents]
    """

    if existing_manual is None:
        existing_manual = {}

    df = pd.read_csv(csv_path, sep="|")

    # Ensure Date column is datetime.date
    df["Date"] = pd.to_datetime(df["Date"]).dt.date

    if start_date:
        start_date = pd.to_datetime(start_date).date()
        df = df[df["Date"] >= start_date]

    if end_date:
        end_date = pd.to_datetime(end_date).date()
        df = df[df["Date"] <= end_date]

    # Drop rows with no assigned resident
    df = df[df["Resident"].notna() & (df["Resident"] != "")]

    # Convert into manual assignment format
    imported_manual = {}

    for _, row in df.iterrows():
        date = row["Date"]
        resident = row["Resident"]

        if date not in imported_manual:
            imported_manual[date] = []

        imported_manual[date].append(resident)

    # Merge with existing_manual
    for date, residents in imported_manual.items():
        if date not in existing_manual:
            existing_manual[date] = residents
        else:
            # Avoid duplicates
            existing_manual[date] = list(set(existing_manual[date] + residents))

    return existing_manual


import_file = output_dir / "import.txt"

if import_file.exists():
    manual_assignments = load_manual_assignments_from_csv(
        import_file,
        existing_manual=manual_assignments
    )
    print("Imported manual assignments:")
else:
    print("No import.txt found — skipping import.")

print(manual_assignments)

### 1a) Define relevant indicies and shifts

In [ ]:
# ------------------------------------------------------------
# MISC INDICES AND CONVENIENCE STRUCTURES FOR LATER PROCESSING
# ------------------------------------------------------------
# Define number of residents and days for later use
n_res = len(residents)

# Define list of all dates in the schedule
dates = [start_date + timedelta(days=i) 
         for i in range((end_date - start_date).days + 1)]
n_days = len(dates)

# Define indices for buddy call period
buddy_indices = [
    d_idx for d_idx, day in enumerate(dates)
    if buddy_call_start <= day <= buddy_call_end
]

# Create mapping of date indices to manual assignments for quick lookup during constraint creation
manual_assignment_indices = {
    d_idx: manual_assignments[day.date()]
    for d_idx, day in enumerate(dates)
    if day.date() in manual_assignments
}

# ------------------------------------------------------------
# BUILD OFFSET MAPS FROM TRANSFERS
# ------------------------------------------------------------
resident_to_idx = {res: i for i, res in enumerate(residents)}

# total shift offset
total_shift_offset = {i: 0 for i in range(n_res)}

# total 12 / 24 shift offsets
total_12_offset = {i: 0 for i in range(n_res)}
total_24_offset = {i: 0 for i in range(n_res)}

# buddy 12 / 24 offsets
buddy_12_offset = {i: 0 for i in range(n_res)}
buddy_24_offset = {i: 0 for i in range(n_res)}

for giver_name, receiver_name, amount, h_filter, b_filter, hol_filter in shift_transfers:
    giver = resident_to_idx[giver_name]
    receiver = resident_to_idx[receiver_name]

    if (h_filter in [12, 24] or h_filter is None):
        # Apply to total for general transfers
        total_shift_offset[receiver] += amount
        total_shift_offset[giver] -= amount

    if h_filter == 12 or h_filter is None:
        total_12_offset[receiver] += amount
        total_12_offset[giver] -= amount

    if h_filter == 24 or h_filter is None:
        total_24_offset[receiver] += amount
        total_24_offset[giver] -= amount


    # Apply to buddy offsets if relevant
    if b_filter is True:
        if h_filter == 12 or h_filter is None:
            buddy_12_offset[receiver] += amount
            buddy_12_offset[giver] -= amount

        if h_filter == 24 or h_filter is None:
            buddy_24_offset[receiver] += amount
            buddy_24_offset[giver] -= amount

# ------------------------------------------------------------
# HALF-DAY PATTERN EXPANSION 
# ------------------------------------------------------------
# Helper: nth weekday of a month (0=Mon ... 6=Sun)
def _nth_weekday(year, month, weekday, nth):
    count = 0
    for day in range(1, 32):
        try:
            d = date(year, month, day)
        except ValueError:
            break
        if d.weekday() == weekday:
            count += 1
            if count == nth:
                return d
    return None

# Expand rotation half-day patterns into actual dates
rotation_halfday_dates = {
    rot: {"AM": [], "PM": []}
    for rot in rotation_half_days
}
for rot, patterns in rotation_half_days.items():
    for block in rotation_blocks:
        # iterate through EVERY month in the block
        current = pd.Timestamp(block["start"]).replace(day=1)
        end = pd.Timestamp(block["end"])

        while current <= end:
            year = current.year
            month = current.month

            for half_type in ["AM", "PM"]:
                for rule in patterns[half_type]:
                    d = _nth_weekday(
                        year,
                        month,
                        rule["weekday"],
                        rule["nth"]
                    )

                    if d and block["start"] <= d <= block["end"]:
                        rotation_halfday_dates[rot][half_type].append(d)

            # move to next month
            current += pd.DateOffset(months=1)

# ------------------------------------------------------------
# OR-DAY PATTERN EXPANSION
# ------------------------------------------------------------
# Expand rotation OR-day patterns into actual dates
rotation_or_dates = {
    rot: []
    for rot in rotation_or_days
}

for rot, patterns in rotation_or_days.items():
    for block in rotation_blocks:

        # iterate through EVERY month in the block
        current = pd.Timestamp(block["start"]).replace(day=1)
        end = pd.Timestamp(block["end"])

        while current <= end:
            year = current.year
            month = current.month

            for rule in patterns:
                d = _nth_weekday(
                    year,
                    month,
                    rule["weekday"],
                    rule["nth"]
                )

                if d and block["start"] <= d <= block["end"]:
                    rotation_or_dates[rot].append(d)

            # move to next month
            current += pd.DateOffset(months=1)
            
# ------------------------------------------------------------
# SHIFT HOURS PER DAY
# ------------------------------------------------------------
# Weekdays = 12h
# Weekends = 24h
# Holidays override to their specified shift length
# ------------------------------------------------------------
shift_hours = []
for d in dates:
    if d in holidays:
        shift_hours.append(holidays[d])  # use override
    elif d.weekday() >= 5:  # Saturday=5, Sunday=6
        shift_hours.append(24)
    else:
        shift_hours.append(12)

# Convenience arrays for fairness logic later
is_24 = np.array([1 if h == 24 else 0 for h in shift_hours])
is_12 = 1 - is_24

# Get indices of full weekend shifts by week number for fairness constraints
full_weekend_by_week = {}
for d_idx, day in enumerate(dates):
    week = day.isocalendar().week
    # Friday (4), Saturday (5), Sunday (6)
    if day.weekday() in (4, 5, 6):
        if week not in full_weekend_by_week:
            full_weekend_by_week[week] = []
        full_weekend_by_week[week].append(d_idx)
# Sorted list of weekend numbers
week_numbers = sorted(full_weekend_by_week.keys())

# Get indices of days by month for fairness constraints
month_indices = defaultdict(list)

for d_idx, day in enumerate(dates):
    month_key = (day.year, day.month)
    month_indices[month_key].append(d_idx)

months_sorted = sorted(month_indices.keys())

# Precompute: which day indices belong to each block
block_day_indices = {b["block"]: [] for b in rotation_blocks}
for b in rotation_blocks:
    block_num = b["block"]
    start = b["start"]
    end = b["end"]
    for d_idx, day in enumerate(dates):
        if start <= day.date() <= end:
            block_day_indices[block_num].append(d_idx)

# Identify holiday indices for 24h and 12h shifts.
holiday_24_indices = [d for d in range(n_days) if dates[d] in holidays and shift_hours[d] == 24]
holiday_12_indices = [d for d in range(n_days) if dates[d] in holidays and shift_hours[d] == 12]

### 1b) Expand and print vacation ranges to include adjacent weekends & holidays

In [ ]:
# ------------------------------------------------------------
# VACATION WEEKEND AND HOLIDAY PADDING
# ------------------------------------------------------------
vacation_block = set()

for entry in vacation_ranges:
    res = entry["res"]
    s = datetime.fromisoformat(entry["start"])
    e = datetime.fromisoformat(entry["end"])

    # Core vacation days
    d = s
    while d <= e:
        vacation_block.add((res, d))
        d += timedelta(days=1)

    # --- Flank before (walk backward through Fri/Sat/Sun/holidays) ---
    d = s - timedelta(days=1)
    while d.weekday() >= 4 or d in holidays:
        vacation_block.add((res, d))
        d -= timedelta(days=1)

    # --- Flank after (walk forward through Fri/Sat/Sun/holidays) ---
    d = e + timedelta(days=1)
    while d.weekday() >= 4 or d in holidays:
        vacation_block.add((res, d))
        d += timedelta(days=1)

# --- DEBUG: Print expanded vacation blocks for verification ---
print("\n============= DEBUG: EXPANDED VACATION RANGES =============\n")

# Group by resident
by_res = defaultdict(list)
for res, day in vacation_block:
    by_res[res].append(day)

def merge_ranges(days):
    days = sorted(days)
    ranges = []
    start = prev = days[0]

    for d in days[1:]:
        if (d - prev).days == 1:
            prev = d
        else:
            ranges.append((start, prev))
            start = prev = d
    ranges.append((start, prev))
    return ranges

# Print compact ranges
for res in sorted(by_res.keys()):
    print(f"=== {res} ===")
    ranges = merge_ranges(by_res[res])
    for s, e in ranges:
        if s == e:
            print(f"  {s.strftime('%Y-%m-%d (%a)')}")
        else:
            print(f"  {s.strftime('%Y-%m-%d (%a)')} → {e.strftime('%Y-%m-%d (%a)')}")
    print()

## 2) Model Parameters -- Adjust This

In [ ]:
# ------------------------------------------------------------
# HARD CONSTRAINT PARAMETERS
# ------------------------------------------------------------
# These parameters define the strict rules that the schedule must follow. Adjusting these will change the feasibility of the schedule and may require more or less manual intervention to find a solution.
# These parameters should generally be left alone.
# ------------------------------------------------------------
# Defeine shift deviation parameters for fairness constraints
max_shift_deviation             = 1 # Allowable deviation in the number of total shifts assigned to each resident (for fairness constraints)
max_12_buddy_shift_deviation    = 1 # Allowable deviation in the number of 12-hour buddy shifts assigned to each senior resident (for fairness constraints)
max_24_buddy_shift_deviation    = 1 # Allowable deviation in the number of 24-hour buddy shifts assigned to each senior resident (for fairness constraints)

# ------------------------------------------------------------
# SOFT CONSTRAINT WEIGHTS
# ------------------------------------------------------------
# These weights can be adjusted to prioritize different aspects of the schedule.
# Higher weights will make the optimizer work harder to satisfy that constraint, while lower weights will make it more likely to violate it if needed to satisfy hard constraints or higher-weighted soft constraints.
# For example, if you want to strongly prioritize avoiding resident preferences, you could increase w_pref_avoid_range. If you want to prioritize spreading out shifts more evenly, you could increase w_weekday_spread and w_month_spread.
# Adjust these weights as needed to find the right balance for your specific scheduling needs.
# Note: It's often helpful to start with all weights equal, then adjust based on the output schedule and which constraints are being violated.
# -------------------------------------------------------------
# Fairness weights (for distributing shifts fairly among residents)
w_shift12_fair      = 5     # Encourage fair distribution of 12-hour buddy shifts among all residents
w_shift24_fair      = 5     # Encourage fair distribution of 24-hour buddy shifts among all residents
w_holiday_fair      = 12    # Encourage fair distribution of holiday shifts among all residents (should be higher than shift fairness weights to prioritize this)

# Diversity weights (for spreading out shifts across different days for each resident)
w_weekday_spread    = 1     # Encourage spreading out shifts across different weekdays for each resident (e.g. not always assigning the same person to Mondays)
w_month_spread      = 1     # Encourage spreading out shifts across different months for each resident (e.g. not always assigning the same person to June)

# Misc preferences
w_near_streak_buddy = 10  
w_near_streak       = 10    # Avoid assigning shifts too close to each other for the same resident (e.g. not assigning someone to two shifts within 2 days of each other, which can be adjusted as needed but should be higher than fairness and diversity weights to prioritize resident well-being)
w_weekend_streak    = 0.01  # Avoid assigning too many weekend shifts consecutively for a resident 
w_pref_avoid_range  = 5     # Avoid assigning residents to shifts during their specified avoid_ranges (should be higher than fairness weights to prioritize resident preferences)
w_tiebreaker        = 0.001 # Small weight to encourage the optimizer to find a solution that satisfies more constraints overall (can be adjusted as needed, but should be smaller than all other weights to ensure it only serves as a tiebreaker)

# Half-day preference weights
w_halfday_PM        = 1     # preference outside buddy-call
w_halfday_AM        = 1     # preference outside buddy-call

# OR day preference weights
w_OR_blackout       = 1

## 3) Build Optimization Model

In [ ]:
# ------------------------------------------------------------
# BUILD OPTIMIZATION MODEL
# ------------------------------------------------------------
model = cp_model.CpModel()

# Decision variable: x[r,d] = 1 if resident r works on day d
x = {}
for r_idx, res in enumerate(residents):
    for d_idx, day in enumerate(dates):
        x[(r_idx, d_idx)] = model.NewBoolVar(f"x_{res}_{d_idx}")

### 3a) Define all hard constraints

In [ ]:
# ------------------------------------------------------------
# NONADJUSTABLE HARD CONSTRAINTS
# ------------------------------------------------------------
# --- Exactly one resident per day ---
for d in range(n_days):
    model.Add(sum(x[(r, d)] for r in range(n_res)) == 1)

# --- Manual assignments ---
for d_idx, allowed_list in manual_assignment_indices.items():
    # 1) Allowed residents: x = 1 is permitted
    # 2) All other residents: x = 0 forced
    for r_idx, res in enumerate(residents):
        if res in allowed_list:
            # allowed → do nothing here
            pass
        else:
            # forbidden → must NOT work this day
            model.Add(x[(r_idx, d_idx)] == 0)
    # 3) Exactly one of the allowed residents must work
    model.Add(
        sum(x[(r_idx, d_idx)]
            for r_idx, res in enumerate(residents)
            if res in allowed_list) == 1
    )

# --- Vacation + flanking weekend/holiday blocks ---
for r_idx, res in enumerate(residents):
    for d_idx, day in enumerate(dates):
        
        # Skip vacation rule if this day has a manual assignment override
        if d_idx in manual_assignment_indices:
            continue

        if (res, day) in vacation_block:
            model.Add(x[(r_idx, d_idx)] == 0)  

# --- Holiday eligibility ---
for d_idx, day in enumerate(dates):
    if day in holidays:
        for r_idx, res in enumerate(residents):
            if res not in residents:
                model.Add(x[(r_idx, d_idx)] == 0)

# --- No resident can work 2 consecutive call shifts ---
# streak[r,d] = 1 if resident r works both day d and day d+1
streak = {}
for r_idx, res in enumerate(residents):
    for d in range(n_days - 1):
        # Forbid working both day d and day d+1
        model.Add(x[(r_idx, d)] + x[(r_idx, d + 1)] <= 1)

# --- No resident may work 5 consecutive full weekends (Fri–Sun) ---
for r_idx, res in enumerate(residents):
    for i in range(len(week_numbers) - 4):
        w1 = week_numbers[i]
        w2 = week_numbers[i + 1]
        w3 = week_numbers[i + 2]
        w4 = week_numbers[i + 3]
        w5 = week_numbers[i + 4]

        idxs = (
            full_weekend_by_week.get(w1, []) +
            full_weekend_by_week.get(w2, []) +
            full_weekend_by_week.get(w3, []) +
            full_weekend_by_week.get(w4, []) +
            full_weekend_by_week.get(w5, []) 
        )

        model.Add(
            sum(x[(r_idx, d)] for d in idxs) <= 4
        )
        
# ------------------------------------------------------------
# ADJUSTABLE HARD CONSTRAINTS
# ------------------------------------------------------------            
# --- Hard fairness for total shifts ---
total_shifts = {}
for r_idx, res in enumerate(residents):
    total_shifts[r_idx] = sum(x[(r_idx, d)] for d in range(n_days))
for r1 in range(n_res):
    for r2 in range(n_res):
        model.Add((total_shifts[r1] - total_shift_offset[r1]) - (total_shifts[r2] - total_shift_offset[r2]) <= max_shift_deviation)
        model.Add((total_shifts[r2] - total_shift_offset[r2]) - (total_shifts[r1] - total_shift_offset[r1]) <= max_shift_deviation)

# --- Buddy-call 12h vs 24h shifts must differ by ≤ max_buddy_shift_deviation ---
buddy_12_count = {}
buddy_24_count = {}
for r_idx, res in enumerate(residents):
    if res in residents:
        buddy_12_count[r_idx] = sum(
            x[(r_idx, d)]
            for d in buddy_indices
            if shift_hours[d] == 12
        )
        buddy_24_count[r_idx] = sum(
            x[(r_idx, d)]
            for d in buddy_indices
            if shift_hours[d] == 24
        )

# Enforce pairwise fairness: difference ≤ max_buddy_shift_deviation
resident_indices = [i for i, r in enumerate(residents) if r in residents]
for i in resident_indices:
    for j in resident_indices:
        # 12-hour buddy-call fairness
        model.Add((buddy_12_count[i] - buddy_12_offset[i]) - (buddy_12_count[j] - buddy_12_offset[j]) <= max_12_buddy_shift_deviation)
        model.Add((buddy_12_count[j] - buddy_12_offset[j]) - (buddy_12_count[i] - buddy_12_offset[i]) <= max_12_buddy_shift_deviation)

        # 24-hour buddy-call fairness
        model.Add((buddy_24_count[i] - buddy_24_offset[i]) - (buddy_24_count[j] - buddy_24_offset[j]) <= max_24_buddy_shift_deviation)
        model.Add((buddy_24_count[j] - buddy_24_offset[j]) - (buddy_24_count[i] - buddy_24_offset[i]) <= max_24_buddy_shift_deviation)

### 3b) Define all soft constraints

In [ ]:
# ------------------------------------------------------------
# HALF-DAY PREFERENCE CONSTRAINTS
# ------------------------------------------------------------
# PM half-day  → prefer call the day BEFORE
# AM half-day  → prefer call the day BEFORE
# ------------------------------------------------------------
halfday_penalties = []
for r_idx, res in enumerate(residents):

    if res not in resident_rotations:
        continue

    for b in rotation_blocks:
        block_num = b["block"]
        rotation_name = resident_rotations[res].get(block_num, None)
        if rotation_name is None:
            continue

        # Get AM/PM half-day dates for this rotation
        half_AM = rotation_halfday_dates[rotation_name]["AM"]
        half_PM = rotation_halfday_dates[rotation_name]["PM"]

        for d_idx in block_day_indices[block_num]:
            day = dates[d_idx]
            day_date = day.date()

            # PM half-day → prefer call ON this day
            if day_date in half_PM:
                prev_idx = d_idx - 1
                if prev_idx < 0:
                    continue

                # Penalize NOT working on the day before this PM half-day
                halfday_penalties.append(
                    w_halfday_PM * (1 - x[(r_idx, prev_idx)])
                )

            # AM half-day → prefer call the day BEFORE
            if day_date in half_AM:
                prev_idx = d_idx - 1
                if prev_idx < 0:
                    continue
                
                # Penalize NOT working the day before an AM half-day
                halfday_penalties.append(
                    w_halfday_AM * (1 - x[(r_idx, prev_idx)])
                )

# ------------------------------------------------------------
# OR-DAY CONSTRAINTS
# ------------------------------------------------------------
# If a resident has an OR day, they SHOULD NOT work the day BEFORE
# ------------------------------------------------------------
or_blackout_penalties = []
for r_idx, res in enumerate(residents):

    if res not in resident_rotations:
        continue

    for b in rotation_blocks:
        block_num = b["block"]
        rotation_name = resident_rotations[res].get(block_num, None)
        if rotation_name is None:
            continue

        or_days = rotation_or_dates.get(rotation_name, [])
        if not or_days:
            continue

        for d_idx in block_day_indices[block_num]:
            day = dates[d_idx]
            day_date = day.date()

            # If this day is an OR day for this resident
            if day_date in or_days:

                prev_idx = d_idx - 1
                if prev_idx < 0:
                    continue

                weight = w_OR_blackout  

                # Add soft penalty: working the day before an OR day is bad
                or_blackout_penalties.append(
                    weight * x[(r_idx, prev_idx)]
                )
                
# ------------------------------------------------------------
# HOLIDAY SHIFT FAIRNESS CONSTRAINTS (TOTAL, 12h, 24h)
# ------------------------------------------------------------
# Holiday shift fairness (senior residents only)
holiday_indices = [d_idx for d_idx, day in enumerate(dates) if day in holidays]

avg_total_holiday   = int(len(holiday_indices) / len(residents)) if residents else 0
avg_holiday_12      = int(len(holiday_12_indices) / len(residents)) if residents else 0
avg_holiday_24      = int(len(holiday_24_indices) / len(residents)) if residents else 0

holiday_dev_pos = {}
holiday_dev_neg = {}
holiday12_dev_pos = {}
holiday12_dev_neg = {}
holiday24_dev_pos = {}
holiday24_dev_neg = {}

for r_idx, res in enumerate(residents):

    holiday_dev_pos[r_idx] = model.NewIntVar(0, 100, f"holiday_dev_pos_{res}")
    holiday_dev_neg[r_idx] = model.NewIntVar(0, 100, f"holiday_dev_neg_{res}")

    holiday12_dev_pos[r_idx] = model.NewIntVar(0, 100, f"holiday12_dev_pos_{res}")
    holiday12_dev_neg[r_idx] = model.NewIntVar(0, 100, f"holiday12_dev_neg_{res}")

    holiday24_dev_pos[r_idx] = model.NewIntVar(0, 100, f"holiday24_dev_pos_{res}")
    holiday24_dev_neg[r_idx] = model.NewIntVar(0, 100, f"holiday24_dev_neg_{res}")

    # total holiday shifts
    total_holiday_for_res = sum(x[(r_idx, d)] for d in holiday_indices)
    model.Add(
        total_holiday_for_res - avg_total_holiday ==
        holiday_dev_pos[r_idx] - holiday_dev_neg[r_idx]
    )

    # holiday 12h fairness
    total_holiday12_for_res = sum(x[(r_idx, d)] for d in holiday_12_indices)
    model.Add(
        total_holiday12_for_res - avg_holiday_12 ==
        holiday12_dev_pos[r_idx] - holiday12_dev_neg[r_idx] 
    )

    # holiday 24h fairness
    total_holiday24_for_res = sum(x[(r_idx, d)] for d in holiday_24_indices)
    model.Add(
        total_holiday24_for_res - avg_holiday_24 ==
        holiday24_dev_pos[r_idx] - holiday24_dev_neg[r_idx]
    )
    
# ------------------------------------------------------------
# 12H vs 24H SHIFT FAIRNESS CONSTRAINTS
# ------------------------------------------------------------
# Balance 12 and 24 hour shifts across all residents
shift12_dev_pos = {}
shift12_dev_neg = {}
shift24_dev_pos = {}
shift24_dev_neg = {}

# Count how many 12h and 24h shifts exist in the schedule
total_12h_shifts = sum(1 for h in shift_hours if h == 12)
total_24h_shifts = sum(1 for h in shift_hours if h == 24)

avg_12h = int(total_12h_shifts / n_res)
avg_24h = int(total_24h_shifts / n_res)

for r_idx, res in enumerate(residents):

    # Total 12h shifts for this resident
    total_12_for_res = sum(
        x[(r_idx, d)] for d in range(n_days) if shift_hours[d] == 12
    )

    # Total 24h shifts for this resident
    total_24_for_res = sum(
        x[(r_idx, d)] for d in range(n_days) if shift_hours[d] == 24
    )

    # Deviation variables
    shift12_dev_pos[r_idx] = model.NewIntVar(0, 200, f"shift12_dev_pos_{res}")
    shift12_dev_neg[r_idx] = model.NewIntVar(0, 200, f"shift12_dev_neg_{res}")

    shift24_dev_pos[r_idx] = model.NewIntVar(0, 200, f"shift24_dev_pos_{res}")
    shift24_dev_neg[r_idx] = model.NewIntVar(0, 200, f"shift24_dev_neg_{res}")

    # Fairness constraints
    model.Add(
        total_12_for_res - avg_12h ==
        shift12_dev_pos[r_idx] - shift12_dev_neg[r_idx]
    )

    model.Add(
        total_24_for_res - avg_24h ==
        shift24_dev_pos[r_idx] - shift24_dev_neg[r_idx]
    )

# ------------------------------------------------------------
# NEAR-STREAK SOFT CONSTRAINT
# ------------------------------------------------------------
# Near streak variables (soft penalty for working d and d+2)
near_streak = {}
for r_idx, res in enumerate(residents):
    for d in range(n_days - 2):
        ns = model.NewBoolVar(f"near_streak_{res}_{d}")
        near_streak[(r_idx, d)] = ns

        # ns = 1 if resident works both day d and day d+2
        model.Add(ns >= x[(r_idx, d)] + x[(r_idx, d + 2)] - 1)
        model.Add(ns <= x[(r_idx, d)])
        model.Add(ns <= x[(r_idx, d + 2)])

near_streak_penalties = []
for r_idx, res in enumerate(residents):
    for d in range(n_days - 2):
        if (buddy_call_start <= dates[d] <= buddy_call_end or
            buddy_call_start <= dates[d+2] <= buddy_call_end):
            weight = w_near_streak_buddy
        else:
            weight = w_near_streak

        near_streak_penalties.append(
            weight * near_streak[(r_idx, d)]
        )

weekend_streak = {}
for r_idx, res in enumerate(residents):
    # Works-any-weekend indicator (Fri–Sun now)
    works_weekend = {
        w: model.NewBoolVar(f"ww_{res}_{w}")
        for w in week_numbers
    }

    for w in week_numbers:
        idxs = full_weekend_by_week.get(w, [])

        if idxs:
            model.AddMaxEquality(
                works_weekend[w],
                [x[(r_idx, d)] for d in idxs]
            )
        else:
            # No weekend days in this week → force 0
            model.Add(works_weekend[w] == 0)

    # Build consecutive-weekend indicators
    for w, w_next in zip(week_numbers, week_numbers[1:]):
        ws = model.NewBoolVar(f"ws_{res}_{w}")
        weekend_streak[(r_idx, w)] = ws

        model.Add(ws >= works_weekend[w] + works_weekend[w_next] - 1)
        model.Add(ws <= works_weekend[w])
        model.Add(ws <= works_weekend[w_next])

# ------------------------------------------------------------
# WEEKDAY SPREAD SOFT CONSTRAINTS
# ------------------------------------------------------------
# Encourage a balanced mix of weekdays. 
weekday_dev_pos = {}
weekday_dev_neg = {}

# Count how many total shifts occur on each weekday
weekday_indices = {w: [] for w in range(7)}  # 0=Mon ... 6=Sun
for d_idx, day in enumerate(dates):
    weekday_indices[day.weekday()].append(d_idx)

# For each weekday, compute average shifts per resident
weekday_avg = {
    w: int(len(weekday_indices[w]) / n_res) if n_res > 0 else 0
    for w in range(7)
}

for r_idx, res in enumerate(residents):
    for w in range(7):
        # deviation variables
        weekday_dev_pos[(r_idx, w)] = model.NewIntVar(0, 100, f"wd_pos_{res}_{w}")
        weekday_dev_neg[(r_idx, w)] = model.NewIntVar(0, 100, f"wd_neg_{res}_{w}")

        # total shifts this resident works on weekday w
        total_w = sum(x[(r_idx, d)] for d in weekday_indices[w])

        # fairness constraint
        model.Add(total_w - weekday_avg[w] ==
                  weekday_dev_pos[(r_idx, w)] - weekday_dev_neg[(r_idx, w)])   

# ------------------------------------------------------------
# MONTH SPREAD SOFT CONSTRAINTS
# ------------------------------------------------------------
# Spread shifts across months
month_dev_pos = {}
month_dev_neg = {}
for r_idx, res in enumerate(residents):

    # Count shifts per month for this resident
    shifts_per_month = []
    for month_key in months_sorted:
        count = sum(x[(r_idx, d)] for d in month_indices[month_key])
        shifts_per_month.append(count)

    # Create deviation variables for month-to-month differences
    for i in range(len(shifts_per_month) - 1):

        dev_pos = model.NewIntVar(0, 100, f"month_dev_pos_{res}_{i}")
        dev_neg = model.NewIntVar(0, 100, f"month_dev_neg_{res}_{i}")

        month_dev_pos[(r_idx, i)] = dev_pos
        month_dev_neg[(r_idx, i)] = dev_neg

        # shifts_month[i] - shifts_month[i+1] = dev_pos - dev_neg
        model.Add(
            shifts_per_month[i] - shifts_per_month[i + 1]
            == dev_pos - dev_neg
        )

# ------------------------------------------------------------
# RESIDENT PREFERENCE SOFT CONSTRAINTS
# ------------------------------------------------------------        
preference_penalties = []

# Soft avoidance of specific date ranges
for r_idx, res in enumerate(residents):
    prefs = resident_preferences.get(res, {})
    avoid_ranges = prefs.get("avoid_ranges", [])

    for d_idx, day in enumerate(dates):
        day_date = day.date()
        for start, end in avoid_ranges:
            if start <= day_date <= end:
                preference_penalties.append(w_pref_avoid_range * x[(r_idx, d_idx)])
                break

### 3c) Define Objective Function and Solve

In [ ]:
# ------------------------------------------------------------
# OBJECTIVE
# ------------------------------------------------------------

# Create the objective function by summing all the components with their respective weights
main_obj = (
    # Fairness for 12h and 24h shifts (all residents)
    + w_shift12_fair    * sum(shift12_dev_pos[r] + shift12_dev_neg[r] for r in range(n_res))
    + w_shift24_fair    * sum(shift24_dev_pos[r] + shift24_dev_neg[r] for r in range(n_res))

    # Holiday shift fairness (senior residents only, should be weighted more than regular shift fairness to prioritize this)
    + w_holiday_fair * (
        sum(holiday_dev_pos[r] + holiday_dev_neg[r] for r in range(n_res))
        + sum(holiday12_dev_pos[r] + holiday12_dev_neg[r] for r in range(n_res))        
        + sum(holiday24_dev_pos[r] + holiday24_dev_neg[r] for r in range(n_res))
    )

    # Diversity of weekdays and months (all residents)
    + w_weekday_spread  * sum(weekday_dev_pos[(r, w)] + weekday_dev_neg[(r, w)] for r in range(n_res) for w in range(7))
    + w_month_spread    * (sum(month_dev_pos.values()) + sum(month_dev_neg.values()))

    # Streak penalties
    + sum(near_streak_penalties)
    + w_weekend_streak  * sum(weekend_streak.values())

    # Half-day preference penalties
    + sum(halfday_penalties)

    # Resident preference penalties
    + sum(preference_penalties)
)

# Add tie breaker penalties
tiebreaker_penalty = []
for r_idx, res in enumerate(residents):
    if res not in tiebreaker_residents:
        # Penalize non-tiebreakers taking extra holiday load
        tiebreaker_penalty.append(holiday_dev_pos[r_idx])
        tiebreaker_penalty.append(holiday12_dev_pos[r_idx])
        tiebreaker_penalty.append(holiday24_dev_pos[r_idx])
main_obj += w_tiebreaker * sum(tiebreaker_penalty)
model.Minimize(main_obj)

# Set up and call the solver
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30

# Solve the model
status = solver.Solve(model)
print("Solver status:", solver.StatusName(status))
schedule = []

# Extract the schedule from the solver's solution
for d_idx, day in enumerate(dates):
    for r_idx, res in enumerate(residents):
        if solver.Value(x[(r_idx, d_idx)]) == 1:
            schedule.append((day.date(), res, shift_hours[d_idx]))

# Convert the schedule to a DataFrame for easier analysis and visualization
df_schedule = pd.DataFrame(schedule, columns=["Date", "Resident", "ShiftHours"])


### 4) Print Results to File

In [ ]:
# ------------------------------------------------------------
# SHIFT SUMMARY TABLE
# ------------------------------------------------------------
# Format:
#   Resident | TotalShifts | 12hShifts | 24hShifts | HolidayShifts | Holiday12hShifts | Holiday24hShifts | BuddyCallShifts | NonBuddyShifts | Buddy12hShifts | Buddy24hShifts
#------------------------------------------------------------
buddy_indices = [
    d_idx for d_idx, day in enumerate(dates)
    if buddy_call_start <= day <= buddy_call_end
]

nonbuddy_indices = [
    d_idx for d_idx, day in enumerate(dates)
    if not (buddy_call_start <= day <= buddy_call_end)
]

holiday_12_indices = [d for d in holiday_indices if shift_hours[d] == 12]
holiday_24_indices = [d for d in holiday_indices if shift_hours[d] == 24]

summary_rows = []

for r_idx, res in enumerate(residents):

    total_shifts = sum(solver.Value(x[(r_idx, d)]) for d in range(n_days))
    buddy_shifts = sum(solver.Value(x[(r_idx, d)]) for d in buddy_indices)
    nonbuddy_shifts = sum(solver.Value(x[(r_idx, d)]) for d in nonbuddy_indices)
    holiday_shifts = sum(solver.Value(x[(r_idx, d)]) for d in holiday_indices)

    shifts_12 = sum(
        solver.Value(x[(r_idx, d)]) 
        for d in range(n_days) if shift_hours[d] == 12
    )
    shifts_24 = sum(
        solver.Value(x[(r_idx, d)]) 
        for d in range(n_days) if shift_hours[d] == 24
    )
    buddy_12 = sum(
        solver.Value(x[(r_idx, d)])
        for d in buddy_indices if shift_hours[d] == 12
    )
    buddy_24 = sum(
        solver.Value(x[(r_idx, d)])
        for d in buddy_indices if shift_hours[d] == 24
    )

    holiday_12 = sum(solver.Value(x[(r_idx, d)]) for d in holiday_12_indices)
    holiday_24 = sum(solver.Value(x[(r_idx, d)]) for d in holiday_24_indices)
    
    summary_rows.append((
        res,            # Resident
        total_shifts,   # TotalShifts
        shifts_12,      # 12hShifts
        shifts_24,      # 24hShifts
        holiday_shifts, # HolidayShifts
        holiday_12,     # Holiday12hShifts
        holiday_24,     # Holiday24hShifts
        buddy_shifts,   # BuddyCallShifts
        nonbuddy_shifts,# NonBuddyShifts
        buddy_12,       # Buddy12hShifts
        buddy_24        # Buddy24hShifts
    ))

df_summary = pd.DataFrame(
    summary_rows,
    columns=[
        "Resident",
        "TotalShifts",
        "12hShifts",
        "24hShifts",
        "HolidayShifts",
        "Holiday12hShifts",
        "Holiday24hShifts",
        "BuddyCallShifts",
        "NonBuddyShifts",
        "Buddy12hShifts",
        "Buddy24hShifts"
    ]
)

# Save to text file
file_path = output_dir / "junior_shift_summary.txt"
df_summary.to_csv(file_path, sep="\t", index=False)

# ------------------------------------------------------------
# MONTH-GRID CALENDAR OUTPUT
# ------------------------------------------------------------
# Format:
#   For each month, a grid with days as rows and resident initials as cell values.
#   Blank cells indicate no assignment; cells with initials indicate assigned resident.
# ------------------------------------------------------------
assignments = {
    row["Date"]: row["Resident"]
    for _, row in df_schedule.iterrows()
}

months = sorted({(d.year, d.month) for d in dates})

calendar_text_output = []

for year, month in months:
    cal = calendar.Calendar(firstweekday=0)
    month_matrix = cal.monthdatescalendar(year, month)

    # Header
    header = f"{year}-{month:02d}\nMon Tue Wed Thu Fri Sat Sun"
    calendar_text_output.append(header)

    # Build rows
    for week in month_matrix:
        row = []
        for day in week:
            if day.month == month and day in assignments:
                row.append(assignments[day].ljust(3))
            elif day.month == month:
                row.append("   ")
            else:
                row.append("   ")
        calendar_text_output.append(" ".join(row))

    calendar_text_output.append("\n")

# Save to text file
file_path = output_dir / "junior_month_grid_calendar.txt"
with open(file_path, "w") as f: f.write("\n".join(calendar_text_output))

# ------------------------------------------------------------
# LONG LIST OUTPUT 
# ------------------------------------------------------------
# Format:
#   Date | Assigned Resident | Holiday? | Vacationing Residents | Half-Day Residents
# ------------------------------------------------------------
long_rows = []

for d_idx, day in enumerate(dates):

    # Assigned resident
    assigned = next(
        (residents[r_idx] for r_idx in range(n_res)
         if solver.Value(x[(r_idx, d_idx)]) == 1),
        None
    )

    # Holiday?
    is_holiday = day in holidays

    # Vacationing?
    vacationing = [res for (res, d) in vacation_block if d == day]

    # Half-day residents
    am_halfday_residents = []
    pm_halfday_residents = []

    day_date = day.date()
    for r_idx, res in enumerate(residents):

        # Which block is this date in?
        block_num = None
        for b in rotation_blocks:
            if b["start"] <= day.date() <= b["end"]:
                block_num = b["block"]
                break

        if block_num is None:
            continue

        # What rotation is this resident on in this block?
        rotation_name = resident_rotations.get(res, {}).get(block_num, None)
        if rotation_name is None:
            continue

        # Get half-day dates for this rotation
        half_AM = rotation_halfday_dates[rotation_name]["AM"]
        half_PM = rotation_halfday_dates[rotation_name]["PM"]

        # If today is an AM half-day for this resident
        if day_date in half_AM:
            am_halfday_residents.append(res)

        # If today is a PM half-day for this resident
        if day_date in half_PM:
            pm_halfday_residents.append(res)

    long_rows.append((
        day.date(),
        assigned,
        is_holiday,
        ", ".join(vacationing) if vacationing else "",
        ", ".join(am_halfday_residents) if am_halfday_residents else "",
        ", ".join(pm_halfday_residents) if pm_halfday_residents else ""
    ))

df_long = pd.DataFrame(
    long_rows,
    columns=["Date", "Resident", "Holiday", "VacationingResidents", "AMHalfDayResidents", "PMHalfDayResidents"]
)

# Display with | delimiters
df_long_display = df_long.copy()
df_long_display["Formatted"] = (
    df_long_display["Date"].astype(str) + " | " +
    df_long_display["Resident"].astype(str) + " | " +
    df_long_display["Holiday"].astype(str) + " | " +
    df_long_display["VacationingResidents"].astype(str) + " | " +
    df_long_display["AMHalfDayResidents"].astype(str) + " | " +
    df_long_display["PMHalfDayResidents"].astype(str)
)

# Save to text file with | delimiter
file_path = output_dir / "junior_long_list_schedule.txt"
df_long.to_csv(file_path, sep="|", index=False)

# ------------------------------------------------------------
# MONTH-GRID CSV OUTPUT (compact table)
# ------------------------------------------------------------
# Format:
#   Day | 2024-01 | 2024-02 | 2024-03 | ...
#   1   | AB      | C       | D       | ...
#   2   | E       | F       |         | ...
# ------------------------------------------------------------

# Build a sorted list of unique (year, month)
months = sorted({(d.year, d.month) for d in dates})

# Create a DataFrame with rows 1–31 and one column per month
month_labels = [f"{year}-{month:02d}" for (year, month) in months]
df_monthgrid = pd.DataFrame(index=range(1, 32), columns=month_labels)

for (year, month) in months:
    col = f"{year}-{month:02d}"
    for day_num in range(1, 32):
        try:
            d = datetime(year, month, day_num)
        except ValueError:
            continue  # skip invalid dates like Feb 30
        key = d.date()
        if key in assignments:
            df_monthgrid.loc[day_num, col] = assignments[key]
        else:
            df_monthgrid.loc[day_num, col] = ""

# Save CSV
file_path = output_dir / "junior_month_grid_compact.csv"
df_monthgrid.to_csv(file_path, index_label="Day")

# ------------------------------------------------------------
# VACATION MONTH-GRID WITH MULTIPLE COLUMNS PER MONTH
# ------------------------------------------------------------
# Format:
#   Day | Jan | Jan | Feb | Feb | Feb | Mar | ...
#   1   | AB  |     | C   |     |     | D   | ...
#   2   |     |     |     |     |     |     |
# ...
# Note: If a month has multiple residents on vacation on the same day, it gets multiple columns (Jan_1, Jan_2, etc.)
# ------------------------------------------------------------
# Convert vacation_block into lookup keyed by datetime.date
vac_lookup = {}
for res, dt in vacation_block:
    vac_lookup.setdefault(dt.date(), []).append(res)

# Build sorted list of months
months = sorted({(d.year, d.month) for d in dates})

# Build dynamic column names (month may need multiple columns)
month_columns = []

# First pass: determine max number of initials per month
month_max = {}
for (year, month) in months:
    max_count = 1
    for day_num in range(1, 32):
        try:
            d = datetime(year, month, day_num).date()
        except ValueError:
            continue
        if d in vac_lookup:
            max_count = max(max_count, len(vac_lookup[d]))
    month_max[(year, month)] = max_count

# Build final column list
for (year, month) in months:
    month_name = calendar.month_name[month]
    count = month_max[(year, month)]
    if count == 1:
        month_columns.append(month_name)
    else:
        for i in range(1, count + 1):
            month_columns.append(f"{month_name}_{i}")

# Create empty DataFrame
df_vacgrid = pd.DataFrame(index=range(1, 32), columns=month_columns)

# Fill grid
col_idx = 0
for (year, month) in months:
    month_name = calendar.month_name[month]
    count = month_max[(year, month)]

    # Determine which columns belong to this month
    if count == 1:
        month_cols = [month_name]
    else:
        month_cols = [f"{month_name}_{i}" for i in range(1, count + 1)]

    for day_num in range(1, 32):
        try:
            d = datetime(year, month, day_num).date()
        except ValueError:
            continue

        if d in vac_lookup:
            initials = sorted(vac_lookup[d])
            # Fill each column with one initial
            for i, col in enumerate(month_cols):
                df_vacgrid.loc[day_num, col] = initials[i] if i < len(initials) else ""
        else:
            # No vacation that day → blank all columns for that month
            for col in month_cols:
                df_vacgrid.loc[day_num, col] = ""

# Save CSV
file_path = output_dir / "junior_vacation_grid.csv"
df_vacgrid.to_csv(file_path, index_label="Day")


### 5) Plot diagnostics of solution for verification and tuning.

In [ ]:
# ------------------------------------------------------------
# BAR CHARTS FOR SHIFT DISTRIBUTION
# ------------------------------------------------------------
df = df_summary.copy()

# Targets
targets = {
    "12hShifts": round(df["12hShifts"].mean()),   # all residents
    "24hShifts": round(df["24hShifts"].mean()),   # all residents

    # seniors only
    "Buddy12hShifts": round(df["Buddy12hShifts"].mean()),
    "Buddy24hShifts": round(df["Buddy24hShifts"].mean()),
    "Holiday12hShifts": round(df["Holiday12hShifts"].mean()),
    "Holiday24hShifts": round(df["Holiday24hShifts"].mean()),
}

fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)
fig.suptitle("Resident Scheduling Fairness", fontsize=18, y=0.98)

def plot_bar(ax, col, target, title, base_color, seniors_only=False):
    values = df[col]

    # Color logic
    colors = []
    colors.append(base_color)

    bars = ax.bar(df["Resident"], values, color=colors)

    # Target line
    line_len = sum(df["Resident"].isin(residents)) / len(df)
    ax.axhline(target, xmax=line_len, color="red", linestyle="--", linewidth=1.5)

    # Zoomed y-axis
    min_val = max(min(values.min(), target), 1)
    max_val = max(values.max(), target)
    ax.set_ylim(min_val - 1, max_val + 1)

    ax.set_title(f"{title} (target={target})")

    # Annotate values
    for i, v in enumerate(values):
        ax.text(i, v, str(int(v)), ha='center', va='bottom', fontsize=9)

# Row 1: All residents
plot_bar(axes[0, 0], "12hShifts", targets["12hShifts"], "12h Shifts", "#4C72B0")
plot_bar(axes[0, 1], "24hShifts", targets["24hShifts"], "24h Shifts", "#DD8452")

# Row 2: Buddy (seniors only)
plot_bar(axes[1, 0], "Buddy12hShifts", targets["Buddy12hShifts"],
         "Buddy 12h Shifts", "#55A868", seniors_only=True)

plot_bar(axes[1, 1], "Buddy24hShifts", targets["Buddy24hShifts"],
         "Buddy 24h Shifts", "#C44E52", seniors_only=True)

# Row 3: Holiday (seniors only)
plot_bar(axes[2, 0], "Holiday12hShifts", targets["Holiday12hShifts"],
         "Holiday 12h Shifts", "#8172B2", seniors_only=True)

plot_bar(axes[2, 1], "Holiday24hShifts", targets["Holiday24hShifts"],
         "Holiday 24h Shifts", "#937860", seniors_only=True)

# Formatting
for ax_row in axes:
    for ax in ax_row:
        x_positions = range(len(df["Resident"]))
        ax.set_xticks(x_positions, labels=df["Resident"])
        ax.tick_params(axis='x', labelbottom=True)
plt.tight_layout()

file_path = output_dir / "junior_scheduling_fairness.png"
plt.savefig(file_path, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ------------------------------------------------------------
# MONTH-GRID BAR CHARTS
# ------------------------------------------------------------
df_plot = df_schedule.copy()
df_plot["Date"] = pd.to_datetime(df_plot["Date"])

# Precompute lookups
shift_set = {(row["Resident"], row["Date"].date()) for _, row in df_plot.iterrows()}
vacation_set = {(r, pd.to_datetime(dt).date()) for r, dt in vacation_block}
holiday_set = {pd.to_datetime(d) for d in holidays.keys()}
months = sorted(df_plot["Date"].dt.to_period("M").unique())

# Grid settings
ncols = 3
nrows = math.ceil(len(months) / ncols)

# Create subplots
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), sharey=True)
axes = axes.flatten()

# Create mapping of resident to y-axis position
res_to_y = {res: i for i, res in enumerate(residents[::-1])} # reverse to have first resident at top

# Function to check if a resident has a half day the next day
def next_day_is_half_day(res, day):
    next_day = (day + pd.Timedelta(days=1)).date()

    # find block for next day
    block_num = None
    for b in rotation_blocks:
        if b["start"] <= next_day <= b["end"]:
            block_num = b["block"]
            break

    if block_num is None:
        return False

    rotation_name = resident_rotations.get(res, {}).get(block_num, None)
    if rotation_name is None:
        return False

    hd = rotation_halfday_dates.get(rotation_name, {})

    return (
        next_day in hd.get("AM", []) or
        next_day in hd.get("PM", [])
    )

# Function to check if a resident has an OR day the next day
def next_day_is_or_day(res, day):
    next_day = (day + pd.Timedelta(days=1)).date()

    # find block for next day
    block_num = None
    for b in rotation_blocks:
        if b["start"] <= next_day <= b["end"]:
            block_num = b["block"]
            break

    if block_num is None:
        return False

    rotation_name = resident_rotations.get(res, {}).get(block_num, None)
    if rotation_name is None:
        return False

    return next_day in rotation_or_dates.get(rotation_name, [])

def has_near_streak(res, day):
    d = day.date()
    return (
        (res, (day + pd.Timedelta(days=2)).date()) in shift_set or
        (res, (day - pd.Timedelta(days=2)).date()) in shift_set
    )

# Plot each month
for i, month in enumerate(months):

    ax = axes[i]
    df_m = df_plot[df_plot["Date"].dt.to_period("M") == month]

    # Month bounds
    year = month.year
    month_num = month.month
    month_start = pd.Timestamp(datetime(year, month_num, 1))
    last_day = calendar.monthrange(year, month_num)[1]
    month_end = pd.Timestamp(datetime(year, month_num, last_day))
    all_days = pd.date_range(month_start, month_end, freq="D")

    # Fixed Y-axis
    ax.set_yticks(range(len(residents)))
    ax.set_yticklabels(residents[::-1])  # reverse to match y mapping
    ax.set_ylim(-0.5, len(residents) - 0.5)

    # Draw vacation layer
    for (res, dt) in vacation_set:
        dt = pd.to_datetime(dt)

        if month_start <= dt <= month_end and res in res_to_y:
            ax.barh(
                res_to_y[res],
                1,
                left=dt,
                color="#7C3AB9",
                alpha=0.4,
                zorder=1
            )

    # Draw holiday overlay 
    all_days = pd.date_range(month_start, month_end, freq="D")
    for d in all_days:
        if d in holiday_set:
            ax.axvspan(
                d,
                d + pd.Timedelta(days=1),
                facecolor="#FFD700",
                edgecolor="none",
                alpha=0.2,
                zorder=0
            )

    # Draw half-days
    for res in residents:
        for d in all_days:
            d_date = d.date()

            # find rotation block
            block_num = None
            for b in rotation_blocks:
                if b["start"] <= d_date <= b["end"]:
                    block_num = b["block"]
                    break

            if block_num is None:
                continue

            rotation_name = resident_rotations.get(res, {}).get(block_num, None)
            if rotation_name is None:
                continue

            hd = rotation_halfday_dates.get(rotation_name, {})

            if d_date in hd.get("AM", []) or d_date in hd.get("PM", []):
                ax.barh(
                    res_to_y[res],
                    1,
                    left=d,
                    facecolor="none",
                    edgecolor="#838383",
                    hatch="...",
                    linewidth=0,
                    zorder=1.5
                )

    # # Draw OR-days (solid outline, different color)
    # for res in residents:
    #     for d in all_days:
    #         d_date = d.date()

    #         # find rotation block
    #         block_num = None
    #         for b in rotation_blocks:
    #             if b["start"] <= d_date <= b["end"]:
    #                 block_num = b["block"]
    #                 break

    #         if block_num is None:
    #             continue

    #         rotation_name = resident_rotations.get(res, {}).get(block_num, None)
    #         if rotation_name is None:
    #             continue

    #         or_days = rotation_or_dates.get(rotation_name, [])

    #         if d_date in or_days:
    #             ax.barh(
    #                 res_to_y[res],
    #                 1,
    #                 left=d,
    #                 facecolor="none",
    #                 edgecolor="#2C3E50",
    #                 hatch="xxx",
    #                 linewidth=0,
    #                 zorder=1.6
    #             )

    # Draw shift bars
    for _, row in df_m.iterrows():
        res = row["Resident"]
        d = row["Date"]

        color = "#1F77B4" if row["ShiftHours"] == 12 else "#27D630"

        # Check flags
        before_halfday = next_day_is_half_day(res, d)
        before_or_day = next_day_is_or_day(res, d)
        near_streak_flag = has_near_streak(res, d)

        # Defaults
        edgecolor = "black"
        linestyle = "-"
        linewidth = 1

        if before_halfday:
            edgecolor = "#976200"   # orange
            linestyle = "--"
            linewidth = 2
        elif before_or_day:
            edgecolor = "#8B0000"   # dark red
            linestyle = "-"
            linewidth = 2.5
        elif near_streak_flag:
            edgecolor = "#FF0000"
            linestyle = "-"
            linewidth = 3

        ax.barh(
            res_to_y[res],
            1,
            left=d,
            color=color,
            edgecolor=edgecolor,
            linestyle=linestyle,
            linewidth=linewidth,
            alpha=0.9,
            zorder=2
        )

        # Buddy boundary line
        buddy_end = pd.to_datetime(buddy_call_end) + pd.Timedelta(days=1)

        if month_start <= buddy_end <= month_end:
            ax.axvline(
                buddy_end,
                color="black",
                linestyle=":",
                linewidth=1
            )

    # Formatting
    year = month.year
    month_name = calendar.month_name[month.month]
    ax.set_title(f"{month_name} {year}", fontsize=12)

    # X-axis
    ax.set_xticks(all_days)
    ax.set_xticklabels([d.day for d in all_days],rotation=90)

    ax.set_xlabel("Day of Month")
    ax.set_ylabel("Resident")

    ax.set_xlim(month_start, month_end + pd.Timedelta(days=1))
    
    # Create a twin axis on top
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(all_days)
    ax_top.set_xticklabels([d.strftime("%a")[0] for d in all_days], fontsize=8)

    # Optional styling
    ax_top.tick_params(axis='x', length=0)  # no tick marks

# Remove extra axes if grid not full
for j in range(len(months), len(axes)):
    fig.delaxes(axes[j])


# Shared legend (top of figure)
legend_handles = [
    mpatches.Patch(color="#1F77B4", label="12h Shift"),
    mpatches.Patch(color="#27D630", label="24h Shift"),
    mpatches.Patch(color="#7C3AB9", alpha=0.5, label="Vacation"),
    mpatches.Patch(color="#FFD700", alpha=0.3, label="Holiday"),

    # Half-day 
    mpatches.Patch(
        facecolor="white",
        edgecolor="#838383",
        hatch="...",
        label="Half Day"
    ),

    # Shift before half-day 
    mpatches.Patch(
        facecolor="white",
        edgecolor="#976200",
        linestyle="--",
        linewidth=2,
        label="Shift Before Half Day"
    ),
    
    # # OR day 
    # mpatches.Patch(
    #     facecolor="white",
    #     edgecolor="#2C3E50",
    #     hatch="xxx",
    #     label="OR Day"
    # ),

    # Shift before OR day
    mpatches.Patch(
        facecolor="white",
        edgecolor="#8B0000",
        linestyle="-",
        linewidth=2.5,
        label="Shift Before OR Day"
    ),

    # Q2 shift
    mpatches.Patch(
        facecolor="white",
        edgecolor="#FF0000",
        linestyle="-",
        linewidth=3,
        label="Q2 Shift"
    ),
]

fig.legend(
    handles=legend_handles,
    loc='upper center',
    ncol=9,
    bbox_to_anchor=(0.5, 0.965)
)

fig.suptitle("Resident Schedule Overview", fontsize=18)
plt.tight_layout(rect=[0, 0, 1, 0.96])

file_path = output_dir / "junior_month_grid_gantt.png"
plt.savefig(file_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
components = {
    # Shift fairness
    "12h Fairness": w_shift12_fair * solver.Value(
        sum(shift12_dev_pos[r] + shift12_dev_neg[r] for r in range(n_res))
    ),
    "24h Fairness": w_shift24_fair * solver.Value(
        sum(shift24_dev_pos[r] + shift24_dev_neg[r] for r in range(n_res))
    ),

    # Holiday fairness (ALL parts combined)
    "Holiday Fairness": w_holiday_fair * solver.Value(
        sum(holiday_dev_pos[r] + holiday_dev_neg[r] for r in range(n_res)) +
        sum(holiday12_dev_pos[r] + holiday12_dev_neg[r] for r in range(n_res)) +
        sum(holiday24_dev_pos[r] + holiday24_dev_neg[r] for r in range(n_res))
    ),

    # Distribution fairness
    "Weekday Spread": w_weekday_spread * solver.Value(
        sum(weekday_dev_pos[(r, w)] + weekday_dev_neg[(r, w)]
            for r in range(n_res) for w in range(7))
    ),
    "Month Spread": w_month_spread * solver.Value(
        sum(month_dev_pos.values()) + sum(month_dev_neg.values())
    ),

    # Near-streak penalty
    "Near-Streak Penalty": w_near_streak * solver.Value(
        sum(near_streak[(r, d)] for r in range(n_res) for d in range(n_days - 2))
    ),

    # OR blackout penalties
    "Halfday Penalty": solver.Value(sum(halfday_penalties)),

    # Preference penalties
    "Preference Penalty": solver.Value(sum(preference_penalties)),
}


# Sort by largest contribution
components = dict(sorted(components.items(), key=lambda x: x[1], reverse=True))

# Colors for readability
color_map = {
    "12h Fairness": "#4C72B0",
    "24h Fairness": "#DD8452",
    "Holiday Fairness": "#C44E52",
    "Weekday Spread": "#55A868",
    "Month Spread": "#8172B2",
    "Near-Streak Penalty": "#937860",
    "Halfday Penalty": "#DA8BC3",
    "Preference Penalty": "#8C8C8C",
}

colors = [color_map[k] for k in components.keys()]

# Plot
plt.figure(figsize=(10, 6))
bars = plt.bar(components.keys(), components.values(), color=colors)

# Add integer labels (important)
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h, f"{int(h)}",
             ha='center', va='bottom')

total_obj = sum(components.values())

plt.title(f"Objective Component Contributions (Total = {int(total_obj)})")
plt.ylabel("Weighted Penalty")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()